# QuadNet Correction — Lid-Driven Cavity

Self-contained notebook that trains a **QuadNet** spatially-dependent
quadratic correction for the lid-driven cavity ROM and visualises the
results.

**Sections:**
1. Imports and setup
2. Model training
3. Evaluation (corrected vs baseline POD-RBF)
4. Comparison plots
5. Correction operator analysis


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from pina import LabelTensor, Trainer
from pina.callbacks import MetricTracker
from pina.loss import LpLoss
from pina.model import FeedForward
from pytorch_lightning.callbacks import EarlyStopping

from problems.setup_cavity import CavityProblem
from nns.quadnet import QuadNet
from rom.corrected_rom import CorrectedROM
from rom.pod_rbf import err, PODRBF
from utils.plotting import plot

os.makedirs("img", exist_ok=True)
print("Imports OK")


## 1. Setup

In [ ]:
# --- Parameters (reduce epochs/train_size for quick testing) ---
FIELD     = "mag(v)"
REDDIM    = 3
TRAIN_SIZE = 200
TEST_SIZE  = 100
EPOCHS     = 200

# --- Load data and build POD + RBF ---
cavity = CavityProblem(FIELD, REDDIM, subset=None,
                       train_size=TRAIN_SIZE, test_size=TEST_SIZE, device="cpu")
data      = cavity.data
pod       = cavity.pod
rbf       = cavity.rbf
params_train     = cavity.params_train
params_test      = cavity.params_test
snapshots_train  = cavity.snapshots_train
snapshots_test   = cavity.snapshots_test
problem  = cavity.problem

print(f"Ndof={cavity.Ndof}  reddim={REDDIM}  "
      f"train={TRAIN_SIZE}  test={TEST_SIZE}")


In [ ]:
# --- Build the QuadNet correction network and CorrectedROM ---
corr_net = QuadNet(cavity.modes, cavity.coords)

rom = CorrectedROM(
    problem=problem,
    reduction_network=pod,
    interpolation_network=rbf,
    correction_network=corr_net,
    loss=LpLoss(relative=True),
)
print(f"ROM created  —  {sum(p.numel() for p in rom.parameters())} trainable params")


## 2. Training

In [ ]:
trainer = Trainer(
    solver=rom,
    max_epochs=EPOCHS,
    accelerator="cpu",
    callbacks=[
        MetricTracker(),
        EarlyStopping(monitor="loss_corr", patience=500,
                      stopping_threshold=1e-2, check_on_train_epoch_end=True),
    ],
    batch_size=TRAIN_SIZE,
)
trainer.train()
rom.eval()
print("Training complete")


## 3. Evaluation

In [ ]:
# --- Corrected ROM errors ---
predicted_train = rom(params_train)
predicted_test  = rom(params_test)

def rel_error(true, pred):
    return (torch.linalg.norm(true - pred, dim=-1)
            / torch.linalg.norm(true, dim=-1)).tensor.cpu().detach().numpy()

corr_train_err = rel_error(snapshots_train, predicted_train)
corr_test_err  = rel_error(snapshots_test,  predicted_test)
print(f"Corrected ROM  —  train: {corr_train_err.mean():.6f} ± {corr_train_err.std():.6f}"
      f"   test: {corr_test_err.mean():.6f} ± {corr_test_err.std():.6f}")


In [ ]:
# --- Baseline POD-RBF errors ---
pod_rbf = PODRBF(pod_rank=REDDIM, rbf_kernel="thin_plate_spline")
pod_rbf.fit(params_train, snapshots_train)

pod_train_pred = pod_rbf(params_train)
pod_test_pred  = pod_rbf(params_test)

pod_train_err = rel_error(snapshots_train, pod_train_pred)
pod_test_err  = rel_error(snapshots_test,  pod_test_pred)
print(f"POD-RBF        —  train: {pod_train_err.mean():.6f} ± {pod_train_err.std():.6f}"
      f"   test: {pod_test_err.mean():.6f} ± {pod_test_err.std():.6f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([corr_train_err, corr_test_err, pod_train_err, pod_test_err],
           tick_labels=["Corr\ntrain", "Corr\ntest", "POD\ntrain", "POD\ntest"],
           patch_artist=True,
           boxprops=dict(facecolor="lightblue"),
           medianprops=dict(color="k"))
ax.set_yscale("log")
ax.set_ylabel("Relative error")
ax.set_title("Corrected ROM vs baseline POD-RBF")
fig.tight_layout()
plt.show()


## 4. Spatial comparison — single test snapshot

In [ ]:
ind = 2
snap    = snapshots_test[ind].tensor.cpu().detach().numpy().ravel()
pred    = predicted_test[ind].tensor.cpu().detach().numpy().ravel()
pod_pred = pod_test_pred[ind].tensor.cpu().detach().numpy().ravel()

fields = [snap, pred, pod_pred, snap - pred, snap - pod_pred]
labels = ["Truth", "Corrected ROM", "POD-RBF",
          "Error corrected", "Error POD"]
plot(data.triang, fields, labels, filename="img/quadnet_mu_cavity_compare.png")


## 5. Correction: approximate vs exact

In [ ]:
exact_corr = CorrectedROM.compute_exact_correction(pod, snapshots_test)
approx_corr = corr_net(params_test, rbf(params_test))

approx = approx_corr[ind].tensor.cpu().detach().numpy().ravel()
exact  = exact_corr[ind].tensor.cpu().detach().numpy().ravel()

fields = [approx, exact, approx - exact]
labels = ["Approx correction", "Exact correction", "Error"]
plot(data.triang, fields, labels, filename="img/quadnet_mu_cavity_correction.png")


## 6. Operator C — entry maps at different $\mu$ values

In [ ]:
mus = torch.tensor([[0.2], [0.4], [0.6], [0.8]])
K = REDDIM * (REDDIM + 1) // 2

for i, mu_val in enumerate(mus):
    mu_batch = LabelTensor(mu_val.unsqueeze(0), ["mu"])
    c = corr_net.C(mu_batch).tensor.cpu().detach().numpy()  # (N_dof, K)
    fields_c = [c[:, j] for j in range(K)]
    labels_c = [f"c{j}" for j in range(K)]
    plot(data.triang, fields_c, labels_c,
         filename=f"img/quadnet_mu_cavity_C_mu{mu_val.item():.1f}.png")


## 7. Operator C — density of entry values

In [ ]:
c_all = corr_net.C(params_test).tensor.cpu().detach().numpy()  # (N_dof, K)

fig, axs = plt.subplots(1, K, figsize=(3 * K, 3))
x_range = np.linspace(c_all.min(), c_all.max(), 200)
for j, ax in enumerate(axs):
    dens = gaussian_kde(c_all[:, j])
    ax.fill_between(x_range, 0, dens(x_range), alpha=0.6, color="grey")
    ax.plot(x_range, dens(x_range), "k")
    ax.set_title(f"C[:, {j}]")
    ax.set_xlabel("value")
axs[0].set_ylabel("density")
fig.suptitle("Distribution of correction-operator entries", y=1.02)
fig.tight_layout()
plt.show()


## 8. Per-sample relative error distribution

In [ ]:
per_sample_err = rel_error(snapshots_test, predicted_test)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(per_sample_err, bins=20, color="grey", edgecolor="k", alpha=0.7)
ax.axvline(per_sample_err.mean(), color="r", ls="--", label=f"mean={per_sample_err.mean():.4f}")
ax.set_xlabel("Relative error")
ax.set_ylabel("Count")
ax.set_title("Test-set relative error distribution")
ax.legend()
fig.tight_layout()
plt.show()


### Done

All sections executed successfully.  To improve accuracy, increase
`EPOCHS` and/or `TRAIN_SIZE` in the setup cell above.
